In [2]:
#FULLY SELF CONTAINED CHECKING OF SEQUENCES MOD PRIMES THAT ARE CONGRUENT TO 1 MOD 11
from itertools import combinations
import pandas as pd

def calculate_xn_mod(a, b, max_n):
    """
    Calculates x_n = a*F_n + b*L_n modulo a specific prime up to max_n
    """
    # Base states for Fibonacci (F) and Lucas (L)
    f_prev, f_curr = 0, 1
    l_prev, l_curr = 2, 1
    
    xn_mod_sequence = []
    
    # Calculate n = 0 manually
    x0 = (a * f_prev + b * l_prev) 
    xn_mod_sequence.append(x0)
    
    # Calculate n = 1 manually
    x1 = (a * f_curr + b * l_curr) 
    xn_mod_sequence.append(x1)
    
    # Loop for subsequent terms
    for n in range(2, max_n + 1):
        # Compute next elements in the base sequences
        f_next = (f_prev + f_curr) 
        l_next = (l_prev + l_curr) 
        
        # Compute x_n modulo m
        xn = (a * f_next + b * l_next) 
        xn_mod_sequence.append(xn)
        
        # Step forward in history
        f_prev, f_curr = f_curr, f_next
        l_prev, l_curr = l_curr, l_next
        
    return xn_mod_sequence



def check_perfect_11(x,prime,exponent): # check if a number is a perfect 11th power mod p
    element_mod_p = Mod(x,prime)
    return pari.ispower(element_mod_p, exponent)

# TO-DO create period length computing function
def compute_period_length(a, b, modulus):
    """
    Computes the period length of the sequence x_n = a*F_n + b*L_n modulo modulus
    Returns the period length and the sequence values
    """
    # Generate sequence until we find a repeat of (x_0, x_1)
    f_prev, f_curr = 0, 1  # F_0, F_1
    l_prev, l_curr = 2, 1  # L_0, L_1
    
    # Calculate initial values
    x0 = (a * f_prev + b * l_prev) % modulus
    x1 = (a * f_curr + b * l_curr) % modulus
    
    # Store pairs (F_n, L_n) or directly store (x_n, x_{n+1})
    sequence_pairs = [(f_prev, l_prev), (f_curr, l_curr)]
    xn_sequence = [x0, x1]
    
    # Keep track of seen states: (F_n, L_n) -> index
    seen_states = {(f_prev, l_prev): 0, (f_curr, l_curr): 1}
    
    n = 1  # We already have n=0 and n=1
    while True:
        n += 1
        
        # Compute next elements
        f_next = (f_prev + f_curr) % modulus
        l_next = (l_prev + l_curr) % modulus
        
        state = (f_next, l_next)
        
        # Check if we've seen this state before
        if state in seen_states:
            # We found the period
            start_index = seen_states[state]
            period_length = n - start_index
            
            # Return all sequence values up to the period
            return period_length
        
        # Store the new state
        seen_states[state] = n
        sequence_pairs.append(state)
        
        # Compute x_n
        xn = (a * f_next + b * l_next) % modulus
        xn_sequence.append(xn)
        
        # Step forward
        f_prev, f_curr = f_curr, f_next
        l_prev, l_curr = l_curr, l_next

def get_eleventh_powers(a,b,primes, max_prime_index):
    power_indices = dict() # to keep which n are perfect 11th powers mod a prime
    primes_to_period = dict() # track period of sequence mod prime
    for p in primes[:max_prime_index]:
        period = compute_period_length(a,b,p)
        seq = calculate_xn_mod(a,b,period-1)
        indices = []
        for el in seq:
            if check_perfect_11(el, p, 11):
                indices.append(seq.index(el))
        if len(indices) == 0:
            print("No 11th powers at", p)
            break
        power_indices[period] = indices
        primes_to_period[period] = p
        print("prime: ", p)
    print("Done computing 11th powers")
    return power_indices, primes_to_period

def chinese_remainder_thm(x,y,m,n):
    if (x - y) % gcd(m,n) == 0:
        return True
    else:
        return False

def check_indices(perfect_powers):
    for key in perfect_powers:
        perfect_powers[key] = [index for index in perfect_powers[key] if index % 2 == 0]
    for key_pair in combinations(perfect_powers, 2):
        print("Working on: ", key_pair)
        if not any(chinese_remainder_thm(n,m, key_pair[0], key_pair[1]) for n in perfect_powers[key_pair[0]] for m in perfect_powers[key_pair[1]]):
            print("Periods of length", key_pair[0], "and", key_pair[1], "do not work")
            return key_pair[0],key_pair[1]
    print('done')

#coefficients of the sequence
a = 42
b = 13

# primes congruent to 1 mod 11
primes = Primes(modulus=11, classes=[1])

perfect_power_indices, period_of_primes = get_eleventh_powers(a,b,primes,10)
print(perfect_power_indices)
check_indices(perfect_power_indices)



prime:  23
prime:  67
prime:  89
prime:  199
prime:  331
prime:  353
prime:  397
prime:  419
prime:  463
prime:  617
Done computing 11th powers
{48: [9, 10, 11, 12, 13, 33, 34, 35, 36, 37], 136: [17, 41, 59, 63, 67, 85, 109, 127, 131, 135], 44: [1, 12, 23, 34], 22: [11], 110: [31, 40, 49, 53, 65, 69, 73, 85, 89, 98, 107], 236: [5, 15, 24, 45, 54, 64, 74, 83, 104, 113, 123, 133, 142, 163, 172, 182, 192, 201, 222, 231], 796: [9, 12, 14, 15, 20, 31, 38, 46, 53, 64, 69, 70, 72, 75, 88, 98, 109, 112, 117, 126, 139, 144, 157, 166, 171, 174, 185, 195, 208, 211, 213, 214, 219, 230, 237, 245, 252, 263, 268, 269, 271, 274, 287, 297, 308, 311, 316, 325, 338, 343, 356, 365, 370, 373, 384, 394, 407, 410, 412, 413, 418, 429, 436, 444, 451, 462, 467, 468, 470, 473, 486, 496, 507, 510, 515, 524, 537, 542, 555, 564, 569, 572, 583, 593, 606, 609, 611, 612, 617, 628, 635, 643, 650, 661, 666, 667, 669, 672, 685, 695, 706, 709, 714, 723, 736, 741, 754, 763, 768, 771, 782, 792], 418: [3, 11, 24, 34, 38, 43,

(48, 136)